In [1]:
from IPython.display import Image

> scaling of (Generalist) reward model

- 两个重点
    - 一个是 **scaling**
        - from one to three scaling laws
            - pre-training
            - post-training
                - RL
            - test-time scaling
                - 根据 query，实时对计算弹性伸缩；
    - 一个是 **general** (通用性)

### SPCT

In [2]:
Image(url='./figs/spct-paradigms.png', width=500)

- (a)(scalar) 不可 inference-time scaling
    - (b) sampling multiples 多次，效果提升不明显，scalar 单独的 head 输出，sample 多次效果比较稳定
- principle => critic => score
    - 奖励过程思维链化（CoT）了
    - （模型自己）定标准，写评价，再打分；
- spct 采用的方式是 (c)+(i)
    - pointwise + generation
    - 注意 pointwise：一个 Pointwise 奖励模型，其最核心的特征是它能够为每一个候选回答 {y_i} 生成一个独立的、绝对的评估分数 {S_i}。
        - input flexibity 更高

### modeling & training

$$
\begin{equation}
    \{S_i\}_{i=1}^n = f_{\text{point}}(R, \{y_i\}_{i=1}^n) = f_{\text{extract}}(C), \quad R = C \sim r_{\theta}(x, \{y_i\}_{i=1}^n), \quad S_i \in \mathbb{R}
\end{equation}
$$
- 模型 $r_{\theta}$ 接收查询 $x$ 和一组回答 $\{y_i\}_{i=1}^n$, 生成文本评判 $C$, 再从中提取出每个回答的数值分数 $S_i$
    - f_point 代表的是 “逐点评分模式” (Pointwise Scoring Pattern)。

自我原则评判调优 (SPCT) 的核心建模，模型不仅生成评判，还自己生成用于评判的“原则”。

$$
\begin{equation}
    \{p_j\}_{j=1}^m \sim p_{\phi}(x, \{y_i\}_{i=1}^n), \quad R = C \sim r_{\theta}(x, \{y_i\}_{i=1}^n, \{p_j\}_{j=1}^m)
\end{equation}
$$
- $p_\theta$ is the principle generation function parameterized by $\theta$, that shares the same model with reward generation $r_\theta$.

#### RFT

$$
\text{if }
\begin{cases}
    \forall i \neq j, \quad S_j > S_i, \quad j = \arg\max_{i'}\{r_{i'}\}_{i'=1}^n, & \text{if } n \ge 2, \\
    S_1 = r_1, & \text{if } n = 1.
\end{cases}
$$
- $r_i$: ground truth reward
- $\{S_i\}_{i=1}^n$: predicetd pointwise rewards
-  当有多个回答时，只有当模型预测的最高分回答与真实最高分回答一致时，这次生成才算正确。当只有一个回答时，预测分数必须与真实分数相等。
-  hinted prompt
    -  we optionally append $\arg\max_{i'}\{r_{i'}\}_{i'=1}^n$ to the prompt of the GRM, termed hinted sampling, expecting the predicted rewards to align with the ground truth..."
    -  we observed that hinted sampled trajectories sometimes **shortcut** the generated critique, especially for reasoning tasks, indicating the necessity and potential benefits of online RL for the GRM.
-  Unlike previous works (...) that mix RM data for single, paired, and multiple responses in different formats, we adopt **pointwise GRM**, introduced in Section 2.1, to flexibly generate **rewards** for **any amount of responses** in the same format."
    -  "Rejective sampled data and RL data are from the same RM datasets, containing the preference for single, paired, and multiple responses, constructed from internal data and open-source datasets..."
-  Hyperparameter
    -  learning rate: $5\times 10^{-6}$
    -  Batch Size: 1024
    -  Training Steps: 900

#### Rule-Based RL 阶段的奖励函数

$$
\begin{equation}
    \hat{r}_i = 
    \begin{cases}
        1, & \text{if } n \ge 2 \text{ and } \forall i' \neq j', S_{j'} > S_{i'}, j' = \arg\max_{i'}\{r_{i'}\}_{i'=1}^n, \\
        1, & \text{if } n = 1 \text{ and } S_1 = r_1, \\
        -1, & \text{otherwise}.
    \end{cases}
\end{equation}
$$

#### RL 阶段的优化目标 (GRPO)

$$
\begin{equation}
    J_{\text{GRPO}}(\theta) = \mathbb{E}_{q \sim P(Q), \{o_i\}_{i=1}^G \sim \pi_{\theta_{\text{old}}}(O|q)} \left[ \frac{1}{G} \sum_{i=1}^G \frac{1}{|o_i|} \sum_{t=1}^{|o_i|} \{\min[\dots] - \beta D_{\text{KL}}[\pi_{\theta}||\pi_{\text{ref}}]\} \right]
\end{equation}
$$
- learning rate: $4\times 10^{-7}$
    - 相比 RFT 阶段，RL 阶段的学习率要小得多，以进行更精细的微调。
- grpo rollout: 4
- kl: 0.08
    - 用于控制策略更新幅度，防止模型遗忘。该值通过在 {0.00, 0.01, 0.04, 0.08} 中进行网格搜索确定。

#### 推理时投票缩放 (Inference-Time Scaling)

$$
\begin{equation}
    S_i^* = \sum_{j=1}^k S_{i,j}, \quad \text{where } \{\{S_{i,j}\}_{i=1}^n = f_{\text{point}}(C_j, \{y_i\}_{i=1}^n)\}_{j=1}^k
\end{equation}
$$
- $k$: 独立采样的次数
- 对同一个查询和一组回答，独立运行 GRM 共 $k$ 次，。由于模型生成的原则和评判具有随机性，每次的结果可能不同。最后将这 $k$ 次得到的分数进行求和（或平均），得到一个更稳健、更细粒度的最终分数。

### 输入输出的角度

- GRM 的输入与传统的奖励模型非常相似，主要包括：
    - 查询 (Query)：用户的提问或指令，在论文中用 $x$ 表示。
    - 一个或多个候选回答 (Responses)：模型针对该查询生成的不同回答，在论文中用 $\{y_i\}$ 表示。
    - 这篇论文的一个创新点在于，他们还引入了一个额外的输入：
        - 原则 (Principles)：这是一组预先定义或由模型自己生成的评估标准/指导原则，在论文中用 $\{p_j\}$ 表示。这些原则告诉模型应该从哪些维度（如准确性、清晰度、安全性等）来评估回答。
- 所以，GRM 的完整输入可以看作是：(查询, 候选回答, 评估原则)。
    - $r_θ(x, {y_i}, {p_j})$

- 这是 GRM 与传统标量奖励模型最根本的区别。GRM 的输出是一段结构化的自然语言文本，这篇论文中称之为**“评判” (Critique)**，用 $C$ 表示。
    - 分析过程 (Analysis)：模型会像一个真正的裁判一样，根据输入的“评估原则”，逐一分析每个“候选回答”的优缺点。
    - 对比和结论 (Comparison & Conclusion)：在分析之后，模型会给出一个明确的对比结论，比如“回答2比回答1更好”。
    - 最终的奖励分数 (Final Scores)：最关键的是，在这段生成的文本末尾，模型会按照一个预设的格式输出最终的打分。

### Meta RM

- 第二层级的、独立的奖励模型，即元奖励模型 (Meta Reward Model, Meta RM) 输出

### Voting

- table 17
    - 模型单次运行的结果可能存在随机性或偏差。为了得到一个更稳定、更可靠的评估，研究人员会让同一个模型（DeepSeek-GRM）对同样的问题和回答，独立地运行多次（在这个表格里是3次，产生了 Result 1, 2, 3）。
    - Result 1 的打分是 \boxed{8, 8}。
        - Response 1 得了 8 分。
        - Response 2 得了 8 分。
    - Result 2 的打分是 \boxed{9, 5}。
        - Response 1 得了 9 分。
        - Response 2 得了 5 分。
    - Result 3 的打分是 \boxed{10, 7}。
        - Response 1 得了 10 分。
        - Response 2 得了 7 分。
    - Response 1 的总票数: 8 (来自Result 1) + 9 (来自Result 2) + 10 (来自Result 3) = 27 分
    - Response 2 的总票数: 8 (来自Result 1) + 5 (来自Result 2) + 7 (来自Result 3) = 20 分
- The input order of responses is reversed for DeepSeek-GRM-27B when generating result 2 and result 3."
    - 这意味着在第2次和第3次运行时，模型看到的“Response 1”其实是我们的“Response 2”，它看到的“Response 2”其实是我们的“Response 1”。这是一种为了避免位置偏见而常用的技巧。